# fold: run without Modal (Colab GPU runtime)

This is the no-Modal path for [MauricioCafiero/fold](https://github.com/MauricioCafiero/fold) --
OpenFold3, RosettaFold3 (RF3), and ESMFold cofolding/structure prediction, running directly on
whatever GPU Colab gives this runtime instead of dispatching to a Modal container.

**⚠️ Experimental / not yet run end-to-end on live Colab.** Everything here mirrors the exact logic
already verified working on Modal (see the main repo README for real smoke-test results), but
Colab's specific CUDA/driver/preinstalled-package versions can shift over time and cause install
hiccups this notebook hasn't been checked against yet. If something breaks, please open an issue
on the repo -- this caveat gets removed once someone's confirmed a clean run.

**Before running anything:** `Runtime` → `Change runtime type` → pick a GPU. A free-tier T4 (16GB)
is enough for ESM2 embeddings, ESMFold, and RF3. OpenFold3's own docs recommend 32GB+ VRAM (it ran
fine on a 24GB A10G in this project's Modal testing) -- free-tier T4 may not be enough for it;
Colab Pro's A100/L4 tiers are a safer bet for that section specifically.

## Check the GPU

In [ ]:
!nvidia-smi

## Setup: clone the repo

Everything below calls into the repo's own code (`fold.openfold3_core`, `fold.rf3_core`,
`fold.esmfold_core`, `fold.local_run`) -- the same logic the Modal apps use, just run in-process
here instead of on a remote container.

In [ ]:
!git clone https://github.com/MauricioCafiero/fold.git
%cd fold
!pip install -q -e .

## Choose a target

Every command below defaults to the built-in smoke test (human HMG-CoA reductase + rosuvastatin,
the same target used in the repo's Modal smoke tests) if you don't pass anything. To use your own:

- `--sequence "..." --smiles "..."` inline, or
- `--input-file path/to/file.txt` with two lines (order doesn't matter):
  ```
  SEQUENCE: MLSRLFRMHGLFVASHPWEVIVG...
  SMILES: CC(C)C1=NC(=NC(=C1)...
  ```

The repo ships `examples/hmgcr_rosuvastatin.txt` in this format, used below.
To use your own file in Colab instead, uncomment and run this cell:

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # then pass --input-file <the filename you uploaded>

## 1. ESM2 embeddings + similarity

No GPU actually required for this one -- included here for completeness since it's part of the
same repo. Small enough to run on CPU if you skip the GPU runtime entirely.

In [ ]:
from fold.embeddings import embed_sequence, cosine_similarity
from fold.targets import HMGCR_1HWL_SEQUENCE

vec = embed_sequence(HMGCR_1HWL_SEQUENCE)
print(f"embedding shape: {vec.shape}")

sim = cosine_similarity(HMGCR_1HWL_SEQUENCE, HMGCR_1HWL_SEQUENCE)
print(f"self-similarity (sanity check, should be ~1.0): {sim:.4f}")

## 2. ESMFold (single-sequence structure prediction)

No MSA step needed -- the simplest of the three GPU tools. Colab's runtime already ships a
CUDA-matched `torch`, so this deliberately does **not** reinstall it (a mismatched `pip install
torch` can break Colab's own CUDA setup) -- only adds `transformers`/`accelerate`.

In [ ]:
!pip install -q transformers accelerate
!python -m fold.local_run esmfold --input-file examples/hmgcr_rosuvastatin.txt

## 3. RosettaFold3 (RF3) cofolding

Fetches its own MSA from the public ColabFold server first (RF3's own CLI has no built-in MSA
search), then runs the fold. Pass `--no-use-msa` to skip that and run single-sequence instead --
much lower confidence, but faster if you're just checking the pipeline works.

In [ ]:
!pip install -q "rc-foundry[rf3]"
!python -m fold.local_run rf3 --input-file examples/hmgcr_rosuvastatin.txt

## 4. OpenFold3 cofolding

Wants more VRAM than the other two (see the GPU note at the top). Also computes its MSA remotely
(ColabFold server), so no local sequence databases needed.

The `typing_extensions` pin below works around a real bug: the published `openfold3` package can
end up with a `typing_extensions` older than what its own `pydantic-core` needs, which breaks
`deepspeed`'s import chain. Same fix used in this repo's Modal app -- see `openfold3_core.py`.

In [ ]:
!pip install -q "openfold3[deepspeed]"
!pip install -q -U "typing_extensions>=4.13"
!python -m fold.local_run openfold3 --input-file examples/hmgcr_rosuvastatin.txt

## Getting your outputs out of Colab

Everything above writes into `outputs/<job_name>/...` inside this Colab VM, which disappears when
the runtime recycles. Zip and download it:

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("fold_outputs", "zip", "outputs")
files.download("fold_outputs.zip")